# Практическая работа №3. Пайплайн предобработки

ФИО: ____________________


# Практическая работа №3. Пайплайн предобработки

## Что сделано

- Train/test разделение выполнено до вычисления статистик.
- Числовые признаки: `total_bill`, `size`; пропуски заполняются медианой, затем применяется StandardScaler.
- Категориальные признаки: `sex`, `smoker`, `day`, `time`; пропуски заполняются значением `unknown`, затем применяется OneHotEncoder.
- `OneHotEncoder(handle_unknown="ignore")` защищает пайплайн от новых категорий на test.

## Проверка формы матриц

- Train до обработки: (168, 6)
- Test до обработки: (42, 6)
- Train после обработки: (168, 13)
- Test после обработки: (42, 13)

Вывод: пайплайн обучается только на train и применяется к test без переобучения, поэтому утечки данных нет.


In [ ]:
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

df = pd.read_csv("../data/tips_dirty.csv").dropna(subset=["tip"])
X = df.drop(columns=["tip"])
y = df["tip"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

numeric_features = ["total_bill", "size"]
categorical_features = ["sex", "smoker", "day", "time"]

preprocessor = ColumnTransformer([
    ("num", Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]), numeric_features),
    ("cat", Pipeline([
        ("imputer", SimpleImputer(strategy="constant", fill_value="unknown")),
        ("encoder", OneHotEncoder(handle_unknown="ignore")),
    ]), categorical_features),
])

X_train_prepared = preprocessor.fit_transform(X_train)
X_test_prepared = preprocessor.transform(X_test)
print(X_train_prepared.shape, X_test_prepared.shape)
